In [0]:
%sql
CREATE OR REPLACE TABLE smart_claims_dev.03_gold.customer_claim_policy_telematics_predicted AS
  SELECT
    t.*,
    a.* EXCEPT (chassis_no, claim_no)
  FROM smart_claims_dev.03_gold.customer_claim_policy_telematics t
  JOIN smart_claims_dev.03_gold.claim_images_predicted a USING (claim_no)

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE smart_claims_dev.02_silver.claims_rules (
  rule_id BIGINT GENERATED ALWAYS AS IDENTITY,
  rule STRING,
  check_name STRING,
  check_code STRING,
  check_severity STRING,
  is_active BOOLEAN
);

In [0]:
def insert_rule(rule, name, code, severity, is_active):
    spark.sql(f"""INSERT INTO smart_claims_dev.02_silver.claims_rules
                  (rule, check_name, check_code, check_severity, is_active)
                  VALUES ('{rule}', '{name}', '{code}', '{severity}', {is_active})""")

In [0]:
valid_policy_date = '''
CASE WHEN to_date(pol_eff_date, "yyyy-MM-dd") <= to_date(claim_date)
      AND to_date(pol_expiry_date, "yyyy-MM-dd") >= to_date(claim_date) THEN "VALID"
     ELSE "NOT VALID"
END
'''
insert_rule('invalid policy date', 'valid_date', valid_policy_date, 'HIGH', True)

exceeds_policy_amount = '''
CASE WHEN CAST(sum_insured AS DOUBLE) >= CAST(total AS DOUBLE) THEN "claim value in the range of premium"
     ELSE "claim value more than premium"
END
'''
insert_rule('exceeds policy amount', 'valid_amount', exceeds_policy_amount, 'HIGH', True)

severity_mismatch = '''
CASE WHEN incident_severity = "Total Loss" AND damage_prediction.label = "major" THEN "Severity matches the report"
     WHEN incident_severity = "Major Damage" AND damage_prediction.label = "minor" THEN "Severity matches the report"
     WHEN (incident_severity = "Minor Damage" OR incident_severity = "Trivial Damage") AND damage_prediction.label = "ok" THEN "Severity matches the report"
     ELSE "Severity does not match"
END
'''
insert_rule('severity mismatch', 'reported_severity_check', severity_mismatch, 'HIGH', True)

exceeds_speed = '''
CASE WHEN telematics_speed <= 45 AND telematics_speed > 0 THEN "Normal Speed"
     WHEN telematics_speed > 45 THEN "High Speed"
     ELSE "Invalid speed"
END
'''
insert_rule('exceeds speed', 'speed_check', exceeds_speed, 'HIGH', True)

release_funds = '''
CASE WHEN reported_severity_check = "Severity matches the report"
      AND valid_amount = "claim value in the range of premium"
      AND valid_date = "VALID" THEN "release funds"
     ELSE "claim needs more investigation"
END
'''
insert_rule('release funds', 'fund_release', release_funds, 'HIGH', True)

In [0]:
from pyspark.sql.functions import expr

df = spark.table("smart_claims_dev.03_gold.customer_claim_policy_telematics_predicted")
rules = spark.sql("SELECT * FROM smart_claims_dev.02_silver.claims_rules WHERE is_active = true ORDER BY rule_id").collect()

for rule in rules:
    print(rule.rule)
    df = df.withColumn(rule.check_name, expr(rule.check_code))

(df.write.mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable("smart_claims_dev.03_gold.claim_insights"))

invalid policy date
exceeds policy amount
severity mismatch
exceeds speed
release funds


In [0]:
%sql
SELECT claim_no, incident_severity, damage_prediction.label AS photo_damage,
       valid_date, valid_amount, reported_severity_check, speed_check, fund_release
FROM smart_claims_dev.03_gold.claim_insights
LIMIT 20

claim_no,incident_severity,photo_damage,valid_date,valid_amount,reported_severity_check,speed_check,fund_release
77cc80fd-6dbe-42ed-a78b-36f364c87d40,Major Damage,minor,VALID,claim value more than premium,Severity matches the report,Normal Speed,claim needs more investigation
c6832700-8f4b-41a0-9d43-06b0b287a495,Total Loss,minor,NOT VALID,claim value in the range of premium,Severity does not match,Normal Speed,claim needs more investigation
7622ba41-57f2-41dd-b31e-b3cc8d211f53,Total Loss,minor,NOT VALID,claim value in the range of premium,Severity does not match,Normal Speed,claim needs more investigation
3b8b7ab9-d050-4641-8cbb-5ab15803e5f6,Total Loss,minor,VALID,claim value more than premium,Severity does not match,Normal Speed,claim needs more investigation
589c91a5-fd4e-4383-a618-9c5d2b706485,Major Damage,minor,VALID,claim value more than premium,Severity matches the report,Normal Speed,claim needs more investigation
d4342039-6f68-4653-88bd-1d319031e1ce,Total Loss,minor,VALID,claim value more than premium,Severity does not match,Normal Speed,claim needs more investigation
f9cc2a2c-7ce8-44f2-be94-5d5ba2d58a2a,Trivial Damage,minor,VALID,claim value in the range of premium,Severity does not match,Normal Speed,claim needs more investigation
1ba885d3-bfa1-460b-ac23-ec5944fb98d4,Total Loss,minor,VALID,claim value more than premium,Severity does not match,Normal Speed,claim needs more investigation
34e5453f-6afb-4aa0-a9aa-a75768bd7d78,Trivial Damage,minor,VALID,claim value more than premium,Severity does not match,Normal Speed,claim needs more investigation
c4359759-004f-4887-ae57-828bee3f523c,Total Loss,minor,VALID,claim value in the range of premium,Severity does not match,Normal Speed,claim needs more investigation


In [0]:
%sql
SELECT fund_release, COUNT(*) AS claims
FROM smart_claims_dev.03_gold.claim_insights
GROUP BY fund_release

fund_release,claims
claim needs more investigation,10527
release funds,2465
